# Data Exploration for Union Elections - Wins/Losses

this is the notebook file used for the Exploratory Data Analysis of the data scraped from NLRB.gov website of different cases. 

We will first go through the raw data looking at the different data types that the features use, and general information that can give us more information about the features before creating visualizations and making preliminary thoughts on what features should be focused on more. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


pd.set_option('display.width', 200)

# same look for every figure: no top/right border, light grid behind the data
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#e6e6e6'
plt.rcParams['axes.axisbelow'] = True

BLUE = '#2a78d6'        # main color
LIGHT_BLUE = '#86b6ef'  # lighter step of the same blue, for ordered pairs (e.g. faster / slower)
ORANGE = '#eb6834'      # contrast color (e.g. won vs lost)
GRAY = '#9a9a9a'        # reference lines and context


df = pd.read_csv("../data/elections_clean.csv")
print(df.head())
# column types and missing values
print(df.info())

In [ ]:
df.dtypes.value_counts()

In [ ]:
df.nunique()

In [ ]:

df[['eligible', 'days_petition_to_election']]. describe()

In [ ]:

df[['employer_has_lawyer', 'petitioner_has_lawyer', 'has_intervenor']].mean()

In [ ]:
df["census_region"].value_counts(normalize=True)

## Now Looking at the Target Variable

Trying to look for information about the number of wins vs the number of losses to see how balanced the datat set is and if theres enough information to make a good prediction for this target variable. 

In [ ]:
print(df['union_won'].value_counts())
print(df['union_won'].value_counts(normalize=True).round(3))

In [ ]:
plt.figure(figsize=(5,2))
df['union_won'].map({1: 'won', 0: 'lost'}).value_counts(normalize=True).plot.barh(color=BLUE)
plt.xlabel('fraction of elections')
plt.ylabel('')
plt.show()

## Looking at individual columns

going one by one looking at each feature in general, sometimes focusing on the relationship to wins. 

In [ ]:
# days from petition to election 
print(df['days_petition_to_election'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

In [ ]:
# long right tail, so log-spaced bins like the capital-gain example in lecture 4
days = df['days_petition_to_election'].dropna()

plt.figure(figsize=(6,3))
days.plot.hist(bins=np.logspace(np.log10(1), np.log10(days.max()), 50), color=BLUE)
plt.semilogx()
plt.xlabel('days from petition to election (log scale)')
plt.ylabel('count')
plt.show()

In [ ]:
# zoom in on the first 90 days, one bar per day
plt.figure(figsize=(8,3))
days.plot.hist(bins=np.arange(0, 91), color=BLUE)
for week in [14, 21, 28, 35, 42]:
    plt.axvline(week + 0.5, color=GRAY, linewidth=0.8, zorder=0)
plt.xlabel('days from petition to election')
plt.ylabel('count')
plt.title('one bar per day; gray lines at 2, 3, 4, 5 and 6 weeks')
plt.show()

print('share of elections a whole number of weeks after the petition:', round((days % 7 == 0).mean(), 3))

looking at size of eligible voters

In [ ]:
print(df['eligible'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

eligible = df['eligible'].dropna()
plt.figure(figsize=(6,3))
eligible.plot.hist(bins=np.logspace(0, np.log10(eligible.max()), 50), color=BLUE)
plt.semilogx()
plt.xlabel('eligible voters in the unit (log scale)')
plt.ylabel('count')
plt.show()

Looking at Ordinal + yes/no features (ones that relate to lawyer/another party participating)

In [ ]:
# Looking at features for if a column has lawyers/how many (ordinal + yes/no features)
for col in ['n_participants', 'n_employer_lawyers', 'n_petitioner_lawyers']:
    plt.figure(figsize=(5,2.5))
    df[col].dropna().astype(int).value_counts().sort_index().plot.bar(color=BLUE)   # sort_index keeps the counts in order
    plt.xlabel(col)
    plt.ylabel('count')
    plt.show()

for col in ['employer_has_lawyer', 'petitioner_has_lawyer', 'has_intervenor']:
    print(df[col].value_counts(normalize=True).round(3), '\n')

Unions

In [ ]:
plt.figure(figsize=(5,8))
df['parent_union'].value_counts().sort_values().plot.barh(color=BLUE)
plt.xlabel('number of elections')
plt.ylabel('')
plt.show()

Regions + states 

In [ ]:
plt.figure(figsize=(5,2.5))
df['census_region'].value_counts().sort_values().plot.barh(color=BLUE)
plt.xlabel('number of elections')
plt.ylabel('')
plt.show()

plt.figure(figsize=(5,5))
df['state'].value_counts().head(20).sort_values().plot.barh(color=BLUE)
plt.xlabel('number of elections')
plt.ylabel('')
plt.title('top 20 states')
plt.show()

print('states/territories:', df['state'].nunique())

The employers and law firms a part of each case

In [ ]:
print(df['employer'].value_counts().head(10))
print('\nStarbucks share of all elections:', round(df['employer'].str.contains('starbucks').mean(), 3))

In [ ]:
# one row per (election, law firm); an election with two firms shows up twice
firm_rows = df[['union_won', 'employer_law_firms']].copy()
firm_rows['firm'] = firm_rows['employer_law_firms'].str.lower().str.split('; ')
firm_rows = firm_rows.explode('firm', ignore_index=True)
# print(firm_rows['firm'].value_counts().head(12))

# the same firm is spelled several ways ("littler mendelson, p.c." / "littler mendelson pc"),
# so group by a keyword in the name, the same idea as parent_union
firm_keywords = {'littler': 'Littler Mendelson', 'ogletree': 'Ogletree Deakins', 'jackson lewis': 'Jackson Lewis',
                 'seyfarth': 'Seyfarth Shaw', 'fisher.*phillips': 'Fisher Phillips', 'morgan.*lewis': 'Morgan Lewis',
                 'constangy': 'Constangy', 'cozen': "Cozen O'Connor", 'fox rothschild': 'Fox Rothschild',
                 'proskauer': 'Proskauer Rose'}

firm_rows['firm_group'] = 'other firm'
for keyword, name in firm_keywords.items():
    firm_rows.loc[firm_rows['firm'].str.contains(keyword, na=False), 'firm_group'] = name
firm_rows.loc[firm_rows['firm'].isna(), 'firm_group'] = 'no firm listed'

plt.figure(figsize=(5,4))
firm_rows['firm_group'].value_counts().sort_values().plot.barh(color=BLUE)
plt.xlabel('number of elections')
plt.ylabel('')
plt.show()

Trends across time : number of elections per month

In [ ]:
per_month = df.groupby(df['election_date'].dt.to_period('M')).size()
# a month with no elections has no row at all, so add every month back in with 0
per_month = per_month.reindex(pd.period_range('2011-10', '2026-09', freq='M'), fill_value=0)
per_month.index = per_month.index.to_timestamp()

plt.figure(figsize=(9,3))
plt.plot(per_month.index, per_month.values, color=BLUE)
plt.ylabel('elections per month')
plt.show()

## Specific features vs the target variable

Days between petition and election

In [ ]:
# category-specific histograms (lecture 4), zoomed to 0-100 days where almost all elections are
won = df[df['union_won'] == 1]['days_petition_to_election'].dropna()
lost = df[df['union_won'] == 0]['days_petition_to_election'].dropna()
print('share of elections within 100 days:', round((days <= 100).mean(), 3))

plt.figure(figsize=(6,3))
plt.hist(won, bins=50, range=(0, 100), density=True, alpha=0.5, color=BLUE, label='won')
plt.hist(lost, bins=50, range=(0, 100), density=True, alpha=0.5, color=ORANGE, label='lost')
plt.legend()
plt.xlabel('days from petition to election (0-100)')
plt.ylabel('density')
plt.show()

# win rate per bin of days
day_bins = [0, 14, 21, 28, 35, 42, 56, 90, 180, 365, 4000]
df['days_bin'] = pd.cut(df['days_petition_to_election'], day_bins)
days_table = df.groupby('days_bin', observed=True)['union_won'].agg(['mean', 'size'])
print(days_table.round(3))

plt.figure(figsize=(7,3))
days_table['mean'].plot.bar(color=BLUE)
plt.axhline(df['union_won'].mean(), color=GRAY, linewidth=1)
plt.ylim(0, 1)
plt.xlabel('days from petition to election')
plt.ylabel('union win rate')
plt.show()

The Size (eligible voters)

In [ ]:
df[['eligible', 'union_won']].boxplot(by='union_won', figsize=(4,3))
plt.semilogy()
plt.suptitle('')
plt.ylabel('eligible voters (log scale)')
plt.show()

size_bins = [0, 5, 10, 20, 50, 100, 250, 500, 1000, 40000]
df['size_bin'] = pd.cut(df['eligible'], size_bins)
size_table = df.groupby('size_bin', observed=True)['union_won'].agg(['mean', 'size'])
print(size_table.round(3))

plt.figure(figsize=(7,3))
size_table['mean'].plot.bar(color=BLUE)
plt.axhline(df['union_won'].mean(), color=GRAY, linewidth=1)
plt.ylim(0, 1)
plt.xlabel('eligible voters')
plt.ylabel('union win rate')
plt.show()

Which Lawyer feature (ordinal features) have the most impact

In [ ]:
lawyer_cols = ['employer_has_lawyer', 'petitioner_has_lawyer', 'has_intervenor']

fig, axes = plt.subplots(1, len(lawyer_cols), figsize=(12, 4), sharey=True)

g = df.groupby('employer_has_lawyer')['union_won'].agg(['mean', 'size'])
axes[0].bar(g.index.map({0.0: 'No Lawyer', 1.0: 'Lawyer'}), g['mean'], color=BLUE)

axes[0].set_title('employer_has_lawyer')
axes[0].set_ylim(0, 1)
    
for i, (m, n) in enumerate(zip(g['mean'], g['size'])):
    axes[0].text(i, m + 0.02, f'{m:.0%}\nn={n}', ha='center', fontsize=8)

g = df.groupby('petitioner_has_lawyer')['union_won'].agg(['mean', 'size'])
axes[1].bar(g.index.map({0.0: 'No Lawyer', 1.0: 'Lawyer'}), g['mean'], color=BLUE)

axes[1].set_title('petitioner_has_lawyer')
axes[1].set_ylim(0, 1)
    
for i, (m, n) in enumerate(zip(g['mean'], g['size'])):
    axes[1].text(i, m + 0.02, f'{m:.0%}\nn={n}', ha='center', fontsize=8)


g = df.groupby('has_intervenor')['union_won'].agg(['mean', 'size'])
axes[2].bar(g.index.map({0.0: 'No Intervenor', 1.0: 'Intervenor'}), g['mean'], color=BLUE)

axes[2].set_title('has_intervenor')
axes[2].set_ylim(0, 1)
    
for i, (m, n) in enumerate(zip(g['mean'], g['size'])):
    axes[2].text(i, m + 0.02, f'{m:.0%}\nn={n}', ha='center', fontsize=8)


axes[0].set_ylabel('union win rate')
plt.tight_layout()
plt.show()
fig.savefig("../figures/sydney_eda/figure_1.png", dpi=300, bbox_inches="tight")


Parent Union vs. Number of Wins

In [ ]:
union_table = df.groupby('parent_union')['union_won'].agg(['mean', 'size']).sort_values('mean')
print(union_table.round(3))

# "Unknown" is the 11 rows where the PDF prints "** Data Incomplete **" instead of a union name
union_table = union_table.drop('Unknown')

plt.figure(figsize=(5,8))
union_table['mean'].plot.barh(color=BLUE)
plt.axvline(df['union_won'].mean(), color='black', linewidth=1)
plt.xlim(0, 1)
plt.xlabel('union win rate (black line = all elections)')
plt.ylabel('')
plt.show()

Regions/where vs. number of wins

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(13, 5))

reg = df.groupby('census_region')['union_won'].mean().sort_values()
ax1.barh(reg.index, reg.values, color=BLUE)
ax1.axvline(df['union_won'].mean(), color=GRAY, ls='--')
ax1.set_title('Win rate by region')
plt.show()
fig.savefig("../figures/sydney_eda/figure_2.png", dpi=300, bbox_inches="tight")

fig, ax2 = plt.subplots(1, 1, figsize=(13, 15))

st = df.groupby('state')['union_won'].agg(['mean', 'size'])
st = st[st['size'] >= 50].sort_values('mean')   # skip tiny states


ax2.barh(st.index, st['mean'], color=LIGHT_BLUE)
ax2.axvline(df['union_won'].mean(), color=GRAY, ls='--')
ax2.set_title('Win rate by state (n ≥ 50)')
plt.show()
fig.savefig("../figures/sydney_eda/figure_3.png", dpi=300, bbox_inches="tight")

The employers law firm 

In [ ]:
firm_table = firm_rows.groupby('firm_group')['union_won'].agg(['mean', 'size']).sort_values('mean')
print(firm_table.round(3))

plt.figure(figsize=(5,4))
firm_table['mean'].plot.barh(color=BLUE)
plt.axvline(df['union_won'].mean(), color='black', linewidth=1)
plt.xlim(0, 1)
plt.xlabel('union win rate')
plt.ylabel('')
plt.show()

Time of year

In [ ]:
recent = df[df['year'] >= 2012]
by_year = recent.groupby('year')['union_won'].mean()
by_year_no_starbucks = recent[~recent['employer'].str.contains('starbucks')].groupby('year')['union_won'].mean()

plt.figure(figsize=(7,3))
plt.plot(by_year.index, by_year.values, marker='o', color=BLUE, label='all elections')
plt.plot(by_year_no_starbucks.index, by_year_no_starbucks.values, marker='o', color=GRAY, label='without Starbucks')
plt.ylim(0.5, 0.9)
plt.ylabel('union win rate')
plt.legend()
plt.show()

plt.figure(figsize=(7,2.5))
recent['year'].value_counts().sort_index().plot.bar(color=BLUE)
plt.ylabel('elections')
plt.show()

In [ ]:
month_table = df.groupby('month')['union_won'].agg(['mean', 'size'])
print(month_table.round(3).T)

plt.figure(figsize=(6,2.5))
month_table['mean'].plot.bar(color=BLUE, rot=0)
plt.axhline(df['union_won'].mean(), color=GRAY, linewidth=1)
plt.ylim(0, 1)
plt.ylabel('union win rate')
plt.show()

## Are any features correlated?

In [ ]:
df['log10_days'] = np.log10(df['days_petition_to_election'])
df['log10_eligible'] = np.log10(df['eligible'])

numeric_cols = ['log10_days', 'log10_eligible', 'n_participants', 'n_employer_lawyers', 'n_petitioner_lawyers',
                'employer_has_lawyer', 'petitioner_has_lawyer', 'has_intervenor', 'year', 'union_won']
corr = df[numeric_cols].corr()

plt.figure(figsize=(6,5))
plt.imshow(corr, cmap='RdBu', vmin=-1, vmax=1)   # blue = positive, red = negative, white = 0
plt.colorbar(label='correlation')
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=90)
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.grid(False)
plt.show()

# heatmap of two continuous columns (lecture 4), on log scales
pair = df[['log10_days', 'log10_eligible']].dropna()
heatmap, xedges, yedges = np.histogram2d(pair['log10_days'], pair['log10_eligible'], bins=40)
heatmap[heatmap == 0] = 0.1   # we take the log and log(0) is undefined

plt.figure(figsize=(5,4))
plt.imshow(np.log10(heatmap).T, origin='lower', vmin=0, cmap='Blues', aspect='auto',
           extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]])
plt.colorbar(label='log10(count)')
plt.xlabel('log10(days from petition to election)')
plt.ylabel('log10(eligible voters)')
plt.grid(False)
plt.show()

sample = df[['log10_days', 'log10_eligible', 'n_participants', 'year']].dropna().sample(3000, random_state=1)
pd.plotting.scatter_matrix(sample, figsize=(8,8), alpha=0.1, s=10, color=BLUE, hist_kwds={'bins': 30, 'color': BLUE})
plt.show()

lawyer_by_year = recent.groupby('year')['employer_has_lawyer'].mean()

plt.figure(figsize=(7,3))
plt.plot(lawyer_by_year.index, lawyer_by_year.values, marker='o', color=BLUE)
plt.ylim(0.5, 1)
plt.ylabel('share of employers with a lawyer')
plt.show()


## Which features matter the most?

Correlation only measures straight-line relationships and doesn't work for categories. A simple
measure that works for every feature is the **spread**: the highest group win rate minus the
lowest, using only groups with at least 100 elections so tiny groups don't win by luck.

Caveat: features with more groups (state has 26) get a bigger spread just by chance, and the
bins I picked affect it. It's a rough ranking, not a test. [did use claude for this]

In [ ]:
def win_rate_spread(column, min_size=100):
    # highest minus lowest win rate over the groups with at least min_size elections
    table = df.groupby(column, observed=True)['union_won'].agg(['mean', 'size'])
    table = table[table['size'] >= min_size]
    return table['mean'].max() - table['mean'].min()

df['n_participants_capped'] = df['n_participants'].clip(upper=8)
df['n_employer_lawyers_capped'] = df['n_employer_lawyers'].clip(upper=4)
df['n_petitioner_lawyers_capped'] = df['n_petitioner_lawyers'].clip(upper=3)

features = {'days to election (10 bins)': 'days_bin',
            'unit size (9 bins)': 'size_bin',
            'employer has lawyer': 'employer_has_lawyer',
            'petitioner has lawyer': 'petitioner_has_lawyer',
            'has intervenor': 'has_intervenor',
            'number of participants': 'n_participants_capped',
            'number of employer lawyers': 'n_employer_lawyers_capped',
            'number of petitioner lawyers': 'n_petitioner_lawyers_capped',
            'parent union (34 groups)': 'parent_union',
            'census region (5 groups)': 'census_region',
            'state (26 with 100+)': 'state',
            'election year': 'year',
            'election month': 'month'}

spread = pd.Series({name: win_rate_spread(col) for name, col in features.items()}).sort_values()
print(spread.round(3))

plt.figure(figsize=(6,4.5))
spread.plot.barh(color=BLUE)
plt.xlabel('highest minus lowest group win rate (groups with 100+ elections)')
plt.show()

## Final Exploratory Visualizations

In [ ]:
def add_headline(headline, subtitle):
    # bold headline (the message) plus a gray subtitle (what is plotted), above the axes
    plt.title(headline, loc='left', fontsize=14, fontweight='bold', pad=30)
    plt.text(0, 1.03, subtitle, transform=plt.gca().transAxes, fontsize=9.5, color='#555555', va='bottom')

finding a trend that unions win more now than in previous years

In [ ]:
win_by_year = recent.groupby('year')['union_won'].mean() * 100

plt.figure(figsize=(8,4.8))
plt.plot(win_by_year.index, win_by_year.values, color=BLUE, linewidth=2, marker='o')
for year in [2012, 2026]:   # label the two ends only
    plt.text(year, win_by_year[year] + 2, f'{win_by_year[year]:.0f}%', ha='center', fontsize=11, fontweight='bold')
plt.ylim(50, 90)
plt.xticks(range(2012, 2027, 2))
plt.grid(axis='x', visible=False)
plt.ylabel('union win rate (%)')
plt.xlabel('year of the election (2026: January to September)')
add_headline('Unions win more often than they did a decade ago',
             'Share of NLRB union elections won per year. The rise is the same without the 725 Starbucks elections.')
plt.savefig('../figures/eda_F_win_rate_by_year.png', dpi=300, bbox_inches='tight')
plt.show()

The longer the wait time is between petition and election, the more likely it is to lose the election

In [ ]:
wait_bins = [0, 14, 21, 42, 90, 365, 4000]
wait_labels = ['≤ 2 weeks', '2–3 weeks', '3–6 weeks', '6 weeks–\n3 months', '3–12\nmonths', '> 1 year']
df['wait'] = pd.cut(df['days_petition_to_election'], wait_bins, labels=wait_labels)
wait_table = df.groupby('wait', observed=True)['union_won'].agg(['mean', 'size'])
print(wait_table.round(3))

tick_labels = [f'{label}\n(n={n:,})' for label, n in zip(wait_table.index, wait_table['size'])]
win_pct = wait_table['mean'] * 100
overall_pct = df['union_won'].mean() * 100

plt.figure(figsize=(8,4.8))
plt.bar(tick_labels, win_pct, color=BLUE, width=0.6)
plt.axhline(overall_pct, color=GRAY, linewidth=1)
plt.text(5.45, overall_pct + 1.5, f'all elections: {overall_pct:.0f}%', ha='right', fontsize=9, color='#555555')
for i, pct in enumerate(win_pct):
    plt.text(i, pct + 1.5, f'{pct:.0f}%', ha='center', fontsize=11, fontweight='bold')
plt.ylim(0, 100)
plt.ylabel('union win rate (%)')
plt.xlabel('time from petition to election')
plt.grid(axis='x', visible=False)
add_headline("The longer the wait, the lower the union's odds",
             'Share of NLRB union elections won, by time from petition to vote (14,560 elections with a petition date)')
plt.savefig('../figures/days_win_rate.png', dpi=300, bbox_inches='tight')
plt.show()

Lawyers and the effect on elections

In [ ]:
lawyer_table = df.groupby(['employer_has_lawyer', 'speed_half'], observed=True)['union_won'].mean().unstack() * 100
lawyer_table.index = ['employer had no lawyer', 'employer had a lawyer']

ax = lawyer_table.plot.bar(color=[BLUE, LIGHT_BLUE], figsize=(7,4.8), rot=0, width=0.7,
                           edgecolor='white', linewidth=2)
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.0f%%', fontsize=11, fontweight='bold')
plt.ylim(0, 100)
plt.ylabel('union win rate (%)')
plt.xlabel('')
plt.grid(axis='x', visible=False)
plt.legend(title='time from petition to vote', loc='upper right', frameon=False)
add_headline('Lawyered-up employers cut union odds, and delay cuts them more',
             'Share of NLRB union elections won, by employer counsel and by wait (split at the 31-day median)')
plt.savefig('../figures/lawyer_and_speed.png', dpi=300, bbox_inches='tight')
plt.show()

How smaller voting pools also indicated a higher chance of winning

In [ ]:
unit_bins = [0, 5, 10, 20, 50, 100, 250, 40000]
unit_labels = ['≤ 5', '6–10', '11–20', '21–50', '51–100', '101–250', '> 250']
df['unit'] = pd.cut(df['eligible'], unit_bins, labels=unit_labels)
unit_table = df.groupby('unit', observed=True)['union_won'].agg(['mean', 'size'])
print(unit_table.round(3))

tick_labels = [f'{label}\n(n={n:,})' for label, n in zip(unit_table.index, unit_table['size'])]
win_pct = unit_table['mean'] * 100

plt.figure(figsize=(8,4.8))
plt.bar(tick_labels, win_pct, color=BLUE, width=0.6)
plt.axhline(overall_pct, color=GRAY, linewidth=1)
for i, pct in enumerate(win_pct):
    plt.text(i, pct + 1.5, f'{pct:.0f}%', ha='center', fontsize=11, fontweight='bold')
plt.ylim(0, 100)
plt.ylabel('union win rate (%)')
plt.xlabel('eligible voters in the unit')
plt.grid(axis='x', visible=False)
add_headline('Small units are the easiest wins',
             'Share of NLRB union elections won, by number of eligible voters (gray line: all elections)')
plt.savefig('../figures/unit_size.png', dpi=300, bbox_inches='tight')
plt.show()